# 01 - U.S. location audit (before cohort filtering)

This is a **location audit**, not an attempt to redo the global EDA or peek at the U.S. holdout's outcomes. It uses the globally cleaned output of the existing `02_clean_data.ipynb`. The raw dataset has no independently verified country column. `City, ST` is a **U.S.-state-code proxy**, not proof that a project was based in the U.S.

Run the original global `02` and `03` first if their artifacts are not available locally.

In [1]:
from pathlib import Path
import sys
HERE = Path.cwd()
PROJECT_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "utils" / "preprocessing.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root or notebooks directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
GLOBAL_DIR = PROJECT_ROOT / "data" / "Processed-Data"
US_DIR = GLOBAL_DIR / "us_only"

import pandas as pd
from utils.us_scope import us_location_audit, us_state_proxy_mask, state_suffix, US_STATE_CODES
source = GLOBAL_DIR / "DSI_kickstarterscrape_dataset_cleaned.csv"
if not source.is_file():
    raise FileNotFoundError(f"Missing {source}; restore the global cleaned CSV from Google Drive or run global notebook 02")
clean = pd.read_csv(source)
assert clean["project id"].is_unique
print("Global cleaned campaigns:", len(clean))

Global cleaned campaigns: 45815


In [2]:
audit, excluded_suffixes = us_location_audit(clean)
display(audit)
display(excluded_suffixes.head(30))

,all_rows,candidate_us_rows,excluded_or_uncertain_rows,missing_locations,no_two_letter_suffix
0,45815,42175,3640,1318,3640


,suffix,n
0,<none>,3640


In [3]:
is_candidate = us_state_proxy_mask(clean)
print("Candidate U.S. state/DC codes:")
display(state_suffix(clean.loc[is_candidate, "location"]).value_counts().head(25).to_frame("campaigns"))
print("Examples excluded due to uncertain locations:")
display(clean.loc[~is_candidate, ["location"]].drop_duplicates().head(25))

Candidate U.S. state/DC codes:


,campaigns
location,
CA,8949
NY,7060
TX,2049
IL,1879
FL,1581
WA,1420
PA,1325
MA,1312
OR,1297


Examples excluded due to uncertain locations:


,location
11,NaN
29,"Taipei, Taiwan"
42,"Oslo, Norway"
58,"Cocham���_, Chile"
107,"Port Au Prince, Haiti"
123,"Toronto, Canada"
139,"Puyo, Ecuador"
165,"St Andrew Parish, Jamaica"
169,"Buenos Aires, Argentina"
184,"Hong Kong, Hong Kong"


## Scope decision

**Check with Parth/Challenge Advisor:** Is the last-two-letter U.S. state/DC proxy acceptable for the Milestone 1 requirement, or is external country-level evidence needed? State-code suffixes and geographic coverage are imperfect. Some U.S. territories, ambiguous locations, and missing locations are excluded by this conservative policy. Do **not** report this as a verified U.S.-only dataset until approved.

`02` creates a separate *provisional* cohort; it does not overwrite the global data.